[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 05](README.md)

# OpenMP target: datos, reducción y fallback

**Tema:** 05 · **Sesiones:** 23, 24 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo minimizar mapeos sin perder coherencia entre host y dispositivo?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Las regiones target deben declarar qué datos llegan, cuáles cambian y cuándo regresan. El fallback verifica lógica, pero no demuestra uso del acelerador.

**Prerrequisitos.**

- OpenMP en CPU y jerarquía de memoria.
- Diferencia entre corrección funcional y evidencia de rendimiento.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Explicar `target`, `teams` y `distribute parallel for`.
- Elegir map/to/from/alloc según el flujo de datos.
- Validar reducción en dispositivo y fallback.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Las regiones de datos persistentes evitan transferencias repetidas cuando varias operaciones reutilizan arreglos.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

`map(to:)` inicializa en dispositivo, `from:` recupera y `tofrom:` realiza ambas direcciones.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Una reducción requiere soporte del compilador/runtime y se valida con una referencia numérica.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- offload — delegación de cómputo a un dispositivo
- mapping — relación entre almacenamiento del host y del dispositivo
- fallback — ejecución alternativa en host que conserva corrección


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Offload Host Device

![Flujo de datos entre host y dispositivo](../../images/offload-host-device.svg)

**Cómo leerlo.** Separa preparación, H2D, kernel, D2H y validación. Esa separación evita llamar tiempo total a una medición que solo cubre el kernel.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "05"
NOTEBOOK = "05_openmp_target/02_openmp_target.ipynb"
assert (ROOT / "curso" / "notebooks" / "05_openmp_target" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Contrato de mapeo

**Situación.** Se deriva la dirección mínima para entradas y salidas de vector add.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
arrays = {"a": {"read": True, "write": False}, "b": {"read": True, "write": False}, "c": {"read": False, "write": True}}
def clause(access):
    if access["read"] and access["write"]: return "tofrom"
    if access["read"]: return "to"
    if access["write"]: return "from"
    return "alloc"
mapping = {name: clause(access) for name, access in arrays.items()}
assert mapping == {"a": "to", "b": "to", "c": "from"}
print(mapping)


### Explicación del resultado

El contrato se revisa cuando un arreglo persiste entre kernels o se inicializa en el dispositivo.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Fallback correcto

**Situación.** Se ejecuta una referencia portable de vector add y se valida elemento a elemento.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
n = 1000
a = [i * 0.5 for i in range(n)]
b = [1.0 - i * 0.25 for i in range(n)]
c = [x + y for x, y in zip(a, b)]
expected = [1.0 + i * 0.25 for i in range(n)]
error = max(abs(x-y) for x, y in zip(c, expected))
assert error < 1e-12
print({"n": n, "max_error": error, "path": "modelo CPU para validar el kernel target"})


### Lectura razonada

La misma entrada y tolerancia se reutilizan cuando el kernel OpenMP target se ejecuta en hardware real.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Cómo demostrarías en el informe que la ejecución ocurrió realmente en un dispositivo?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Implementar vector add con región de datos explícita.
2. Reutilizar datos durante varias operaciones y medir ambas variantes.
3. Registrar compilador, plugin de offload y dispositivo.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Mapear `tofrom` todo por comodidad.
- Acceder en host antes de sincronizar.
- Declarar éxito GPU sin comprobar `omp_is_initial_device`.


## Criterios de aceptación

- Mapeo mínimo justificado.
- Fallback y dispositivo producen resultados equivalentes.
- Informe identifica inequívocamente dónde se ejecutó.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo minimizar mapeos sin perder coherencia entre host y dispositivo?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Planeación target](../../../docs/PLANEACION_CURSO.md)
- [Protocolo](../../../docs/REPRODUCIBILIDAD_EJERCICIOS.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 05](README.md)
